In [1]:
import torch
import torch.nn as nn

# 하이퍼파라미터 정의 (다층 + 양방향 모델 구조 강제)
seq_len = 5
batch_size = 2
input_size = 4
hidden_size = 3
num_layers = 2

# Bidirectional 모델 선언
bi_lstm = nn.LSTM(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=True
)

# 더미 시퀀스 데이터 생성 (seq_len, batch_size, input_size)
X = torch.randn(seq_len, batch_size, input_size)

# 순전파 실행
output, (h_n, c_n) = bi_lstm(X)

print("=== [실험 2] 텐서 기하학적 슬라이싱 매치 ===")
print(f"Output Shape : {list(output.shape)}")  # [5, 2, 6] -> 6 = 2(방향) * 3(hidden)
print(f"H_n Shape    : {list(h_n.shape)}")     # [4, 2, 3] -> 4 = 2(레이어) * 2(방향)

print("\n--- 수학적/물리적 일치성 검증 ---")

# 가설: output의 최후 타임스텝(T=-1) 내 정방향(Forward) 벡터는
#      h_n의 최상위 레이어(Layer 2)의 정방향 인덱스 메모리와 완벽히 일치해야 한다.
# PyTorch 정방향/역방향 레이어 배치 규칙에 따라 h_n의 마지막 레이어 정방향 인덱스는 -2 (또는 2)
top_layer_forward_hn = h_n[2, :, :]

# output 내부에서 최상위 레이어의 정방향 컴포넌트 추출 (인덱스 0:hidden_size)
last_timestep_output_forward = output[-1, :, :hidden_size]

# 물리적 일치도 검증 (Floating point 오차 감안)
is_matching = torch.allclose(top_layer_forward_hn, last_timestep_output_forward, atol=1e-6)
print(f"▶ 최상위 레이어 [정방향] 일치 여부: {is_matching}")

# 가설 2: 역방향(Backward)의 최종 상태는 시퀀스의 시작점(T=0)에 기록된다.
# h_n의 최상위 레이어 역방향 인덱스는 -1 (또는 3)
top_layer_backward_hn = h_n[3, :, :]
first_timestep_output_backward = output[0, :, hidden_size:]

is_backward_matching = torch.allclose(top_layer_backward_hn, first_timestep_output_backward, atol=1e-6)
print(f"▶ 최상위 레이어 [역방향] 일치 여부: {is_backward_matching}")

=== [실험 2] 텐서 기하학적 슬라이싱 매치 ===
Output Shape : [5, 2, 6]
H_n Shape    : [4, 2, 3]

--- 수학적/물리적 일치성 검증 ---
▶ 최상위 레이어 [정방향] 일치 여부: True
▶ 최상위 레이어 [역방향] 일치 여부: True
